# Veto DAQ

按功能组织协议、器件配置和硬件测试。函数代码块只定义接口，测试代码块才实际访问板卡。

## 1. 环境与 UDP/RBCP 基础

### 环境导入与全局变量

导入 Python 依赖，不访问硬件。

In [314]:
import serial
import time
import numpy as np
import re

import socket
import os
import binascii
import struct

import matplotlib.pyplot as plt

import serial.tools.list_ports

### UDP/RBCP 基础函数

包含 UDP 配置、单字节写入、慢控返回读取和 stage2 flush。只定义函数。

In [326]:
def udp_config(
    CMD, stage1_addr, underwater_addr, data=None,
    udp_ip='192.168.31.16', udp_port=4660, timeout=5.0,
    verbose=False
):
    if CMD == 'W':
        if data is None:
            raise ValueError("CMD='W' requires data")
        if isinstance(data, int):
            data = bytes([data])
        elif isinstance(data, (list, tuple)):
            data = bytes(data)
        cmd_byte = 0x80
        payload = data
    elif CMD == 'R':
        cmd_byte = 0xC0
        payload = b''
    else:
        raise ValueError("CMD must be 'W' or 'R'")

    packet = bytes([
        0xFF, cmd_byte, 0x00, 0x01,
        0x00, 0x00, underwater_addr, stage1_addr
    ]) + payload

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.settimeout(timeout)

    try:
        sock.bind(('', udp_port))
        sock.sendto(packet, (udp_ip, udp_port))

        try:
            raw, addr = sock.recvfrom(1024)
        except socket.timeout:
            raise RuntimeError(
                f"UDP response timeout: target={udp_ip}:{udp_port}"
            )

        if verbose:
            print(f"RX from {addr}, len={len(raw)}, raw={raw.hex(' ')}")

        if len(raw) < 8:
            raise RuntimeError(
                f"Invalid RBCP response length: {len(raw)}, raw={raw.hex(' ')}"
            )

        length = int.from_bytes(raw[2:4], 'big')
        if len(raw) < 8 + length:
            raise RuntimeError(
                f"Invalid RBCP payload length: {len(raw)}, expected >= {8 + length}, "
                f"raw={raw.hex(' ')}"
            )

        header = raw[0]
        cmd_ack = raw[1]

        if header != 0xFF:
            raise RuntimeError(f"Invalid RBCP header: {raw.hex(' ')}")

        is_ack = (cmd_ack >> 3) & 1
        bus_err = cmd_ack & 1

        if not is_ack:
            raise RuntimeError(f"RBCP ACK bit not set: {raw.hex(' ')}")

        if bus_err:
            raise RuntimeError(f"RBCP bus error: {raw.hex(' ')}")

        return raw, raw[8:8 + length]

    finally:
        sock.close()


def udp_write_byte(value, stage1_addr, underwater_addr,
                   udp_ip='192.168.31.16', udp_port=4660, timeout=5.0):
    """Write one slow-control byte and wait for its RBCP ACK."""
    return udp_config(
        'W', stage1_addr, underwater_addr, value,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )


def udp_read_word(stage1_addr, underwater_addr,
                  udp_ip='192.168.31.16', udp_port=4660, timeout=5.0):
    """
    Collect one 16-bit slow-control result.

    stage2's asymmetric 16-to-8-bit FIFO emits the high byte first.
    Exactly two RBCP reads are required to keep the FIFO aligned.
    Before reading, poll RBCP address high byte 0xFA until a slow-control
    result is available. This avoids a timeout returning a fake 0x00 and
    corrupting the high/low byte pairing.
    """
    status_addr = 0xFA
    deadline = time.monotonic() + timeout

    while True:
        _raw_status, status_payload = udp_config(
            'R', stage1_addr, status_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
        )
        if len(status_payload) != 1:
            raise RuntimeError(
                f"Unexpected slow-control status length: {len(status_payload)}"
            )
        if status_payload[0] & 0x01:
            break
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"Slow-control result timeout: stage1={stage1_addr}, "
                f"underwater={underwater_addr}"
            )
        time.sleep(0.005)

    raw_hi, payload_hi = udp_config(
        'R', stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    raw_lo, payload_lo = udp_config(
        'R', stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )

    if len(payload_hi) != 1 or len(payload_lo) != 1:
        raise RuntimeError(
            "Unexpected slow-control read length: "
            f"hi={len(payload_hi)}, lo={len(payload_lo)}"
        )

    return (payload_hi[0] << 8) | payload_lo[0]

def udp_flush_slow_control(stage1_addr,
                            udp_ip='192.168.31.16', udp_port=4660,
                            timeout=5.0):
    """Discard stale slow-control response bytes in stage2.

    The FPGA waits until both its response FIFO and the stage1-to-stage2
    return path have been quiet before acknowledging this command.
    """
    _raw, payload = udp_config(
        'R', stage1_addr, 0xFB,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    if len(payload) != 1:
        raise RuntimeError(
            f'Unexpected slow-control flush response: {payload!r}'
        )


DAQ_RBCP_API_VERSION = 4

### RBCP 原始写入测试

直接发送一个 W 包，用于检查最基本的 SiTCP/RBCP ACK 链路。

In [278]:
udp_config('W', 1, 1, 0XF0)

(b'\xff\x88\x00\x01\x00\x00\x01\x01\xf0', b'\xf0')

## 2. PTP 时钟同步与 CH0 复位

### PTP 控制、复位和读取函数

包含 PTP start、结果读取、flush，以及按 stage1/underwater 地址选择通道的零延时复位 `ptp_reset()`。只定义函数。

In [327]:
PTP_RESULT_BITS = 384
PTP_RESULT_BYTES = PTP_RESULT_BITS // 8
PTP_API_VERSION = 10


def ptp_buffer_reset(stage1_addr=1, channel=1,
                     udp_ip='192.168.31.16', udp_port=4660,
                     timeout=5.0):
    """Reset stage1 PTP output FIFOs for one selected channel."""
    if not 1 <= channel <= 8:
        raise ValueError('channel must be in 1..8')
    cmd_addr = 0xC0 | channel
    udp_config(
        'W', stage1_addr, cmd_addr, 0x00,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    time.sleep(0.002)


def ptp_flush(stage1_addr=1, channel=1,
              udp_ip='192.168.31.16', udp_port=4660,
              timeout=5.0, max_bytes=2048):
    """Drain stale PTP bytes before starting a new transaction."""
    ptp_buffer_reset(
        stage1_addr=stage1_addr, channel=channel,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    channel_rd_addr = 0xF0 | channel
    drained = bytearray()

    for _ in range(max_bytes):
        _raw_status, status = udp_config(
            'R', stage1_addr, 0xF9,
            udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
        )
        if len(status) != 1:
            raise RuntimeError(
                f"Unexpected PTP status payload length: {len(status)}"
            )
        if not (status[0] & 0x01):
            break

        _raw_data, payload = udp_config(
            'R', stage1_addr, channel_rd_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
        )
        if len(payload) != 1:
            raise RuntimeError(
                f"Unexpected PTP data payload length: {len(payload)}"
            )
        drained.extend(payload)

    return bytes(drained)


def ptp_start(stage1_addr=1, channel=1,
              udp_ip='192.168.31.16', udp_port=4660,
              timeout=5.0):
    """Send PTP start and wait for the stage2 RBCP ACK."""
    if not 1 <= stage1_addr <= 8:
        raise ValueError("stage1_addr must be in 1..8")
    if not 1 <= channel <= 8:
        raise ValueError("channel must be in 1..8")

    ptp_control_addr = 0xF0 | channel
    raw_ack, payload = udp_config(
        'W', stage1_addr, ptp_control_addr, 0x00,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    print(
        f"PTP start ACK: stage1={stage1_addr}, "
        f"channel={channel}, cmd_addr=0x{ptp_control_addr:02X}"
    )
    return raw_ack, payload


def ptp_reset(stage1_addr=1, underwater_addr=1,
              udp_ip='192.168.31.16', udp_port=4660,
              timeout=5.0):
    """Reset one underwater PTP state with a zero-delay T5 transaction.

    underwater_addr selects stage1 channel 1~8.  The stage1-local command
    0xD1~0xD8 drives ptp_value=0 with ptp_value_valid high, issues
    ptp_start, then drops ptp_value_valid.  No normal PTP result is queued.
    """
    if not isinstance(underwater_addr, int) or not (1 <= underwater_addr <= 8):
        raise ValueError('underwater_addr must be an integer between 1 and 8')

    cmd_addr = 0xD0 | underwater_addr
    _raw, _payload = udp_config(
        'W', stage1_addr, cmd_addr, 0x00,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    print(
        f'PTP reset ACK: stage1={stage1_addr}, '
        f'underwater={underwater_addr}, cmd_addr=0x{cmd_addr:02X}'
    )


def ptp_read_data(stage1_addr=1, channel=1,
                  max_bytes=PTP_RESULT_BYTES,
                  udp_ip='192.168.31.16', udp_port=4660,
                  timeout=5.0, idle_timeout=1.0,
                  poll_interval=0.01):
    """
    Read all currently available PTP bytes through non-blocking UDP status
    queries, followed by one-byte RBCP reads.

    RBCP address high byte:
      F1..F8 : pop one byte from fifo_ptp
      F9     : query whether fifo_ptp has at least one byte

    Reading stops after max_bytes, or after fifo_ptp remains empty for
    idle_timeout seconds. This avoids leaving stage2 stuck in RD_WAIT.
    """
    if not 1 <= stage1_addr <= 8:
        raise ValueError("stage1_addr must be in 1..8")
    if not 1 <= channel <= 8:
        raise ValueError("channel must be in 1..8")
    if max_bytes <= 0:
        raise ValueError("max_bytes must be positive")

    ptp_control_addr = 0xF0 | channel
    ptp_status_addr = 0xF9
    data = bytearray()
    idle_since = None

    while len(data) < max_bytes:
        _raw_status, status_payload = udp_config(
            'R', stage1_addr, ptp_status_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
        )
        if len(status_payload) != 1:
            raise RuntimeError(
                f"Unexpected PTP status payload length: {len(status_payload)}"
            )

        if status_payload[0] & 0x01:
            idle_since = None
            _raw_data, payload = udp_config(
                'R', stage1_addr, ptp_control_addr,
                udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
            )
            if len(payload) != 1:
                raise RuntimeError(
                    f"Unexpected PTP data payload length: {len(payload)}"
                )
            data.extend(payload)
        else:
            now = time.monotonic()
            if idle_since is None:
                idle_since = now
            elif now - idle_since >= idle_timeout:
                break
            time.sleep(poll_interval)

    result = bytes(data)
    print(
        f"PTP data: stage1={stage1_addr}, channel={channel}, "
        f"{len(result)} bytes = {len(result) * 8} bits"
    )
    return result


def ptp_result_delay_ns(data):
    """Return the coarse-time delay encoded by a 48-byte PTP result."""
    if len(data) != PTP_RESULT_BYTES:
        return None
    coarse = [
        int.from_bytes(data[offset:offset + 4], 'big') >> 8
        for offset in (32, 36, 40, 44)
    ]
    return ((coarse[1] - coarse[0]) -
            (coarse[3] - coarse[2])) / 2 * 2.5


def ptp_start_and_read(stage1_addr=1, channel=1,
                       max_bytes=PTP_RESULT_BYTES,
                       udp_ip='192.168.31.16', udp_port=4660,
                       timeout=5.0, idle_timeout=1.0):
    """Run exactly one PTP transaction and return its raw bytes unchanged."""
    ptp_flush(
        stage1_addr=stage1_addr, channel=channel,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    ptp_start(
        stage1_addr=stage1_addr, channel=channel,
        udp_ip=udp_ip, udp_port=udp_port, timeout=timeout
    )
    data = ptp_read_data(
        stage1_addr=stage1_addr, channel=channel,
        max_bytes=max_bytes,
        udp_ip=udp_ip, udp_port=udp_port,
        timeout=timeout, idle_timeout=idle_timeout
    )
    return data, len(data) * 8


### PTP 单通道零延时复位测试

按 `stage1_addr` 和 `underwater_addr` 选择通道，执行一次 reset：`ptp_value=0`、`ptp_value_valid=1`、`ptp_start` 脉冲，然后释放 valid。该操作不读取普通 PTP 结果。

In [317]:
ptp_reset(stage1_addr=1, underwater_addr=1)


PTP reset ACK: stage1=1, underwater=1, cmd_addr=0xD1


### PTP 单次同步与结果读取测试

发送一次 PTP start，并读取 stage2 中返回的完整 48 字节结果。

In [339]:
ptp_bytes, ptp_bits = ptp_start_and_read(
    stage1_addr=1,
    channel=1
)
print(ptp_bytes.hex(' '))

result = ptp_bytes

t = []
i = 0
arr_timestamp = np.frombuffer(result[48*i:48*i+32], dtype='>u8')#.astype(int)
arr_tdc = np.frombuffer(result[48*i+32:48*i+48], dtype='>u4')#.astype(int)
t.append(arr_timestamp.tolist() + (arr_tdc >> 8).tolist() + (arr_tdc&0xff).tolist())
t = np.array(t).squeeze()
print('ptp timestamps [s]:', t[:4]*8e-9)
print('ptp coarse time [ms]:', t[4:8]*2.5e-6)

dt = ((t[5] - t[4]) - (t[7] - t[6]))/2 *2.5
print('coarse time delay:', dt, 'ns')

PTP start ACK: stage1=1, channel=1, cmd_addr=0xF1
PTP data: stage1=1, channel=1, 48 bytes = 384 bits
00 00 01 0c dc c3 e4 f8 00 00 00 03 79 c4 55 21 00 00 00 03 79 c4 55 23 00 00 01 0c dc c3 e5 27 b1 12 8d 8f b1 13 3c 63 43 d9 70 77 00 00 00 00
ptp timestamps [s]: [9238.04048787  119.42249498  119.422495   9238.04048825]
ptp coarse time [ms]: [29.0115525 29.01199   11.11644    0.       ]
coarse time delay: 5558438.75 ns


## 3. Si5345 时钟芯片

### Si5345 配置、写入和单寄存器读取函数

解析 ClockBuilder 文件、配置芯片并读取单个寄存器。只定义函数。

In [291]:
def parse_si5345_txt_file(file_path):
    """
    Parse Si5345 register config text file.

    Returns
    -------
    config_data : bytearray
        Byte array of SPI data, 4 bytes per register:
        [0x00, addr, 0x40, data] with page-switch commands inserted.
    """
    result = bytearray()
    current_page = None
    spi_3wire_value = 0x08

    with open(file_path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line or line.startswith('#') or ',' not in line:
            continue
        try:
            addr_data = line.split(',')
            address = int(addr_data[0].strip(), 16)
            data = int(addr_data[1].strip(), 16)
        except (ValueError, IndexError):
            continue

        if address == 0x002B:
            spi_3wire_value = data | 0x08

        page = (address >> 8) & 0xFF
        addr = address & 0xFF

        if page != current_page:
            result.extend([0x00, 0x01, 0x40, page])
            current_page = page

        result.extend([0x00, addr, 0x40, data])

    result.extend([0x00, 0x01, 0x40, 0x00])
    result.extend([0x00, 0x2B, 0x40, spi_3wire_value])
    return result

def config_si5345(config_data, stage1_addr, underwater_addr,
                   udp_ip='192.168.31.16', udp_port=4660):
    """
    Configure one Si5345 on a given stage1+underwater board.

    Sends the 0xF0 bulk-write command followed by length and config
    bytes one at a time over UDP slow control.

    Parameters
    ----------
    config_data : bytearray
        Output of parse_si5345_txt_file.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """
    length = len(config_data)

    packet_count = 0
    prev_ack_ok = True  # 上一个包是否已收到ACK
    # Remove any response left by an aborted or previous transaction.
    udp_flush_slow_control(
        stage1_addr, udp_ip=udp_ip, udp_port=udp_port
    )

    def _send_byte(b):
        nonlocal packet_count, prev_ack_ok
        assert prev_ack_ok, "上一个包未收到ACK，不能发送下一个包"
        packet_count += 1
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port, timeout=5.0
        )
        prev_ack_ok = True  # udp_config 阻塞等待ACK，返回即表示成功

    # 0xF0 command byte
    _send_byte(0xF0)

    # 2-byte big-endian length
    _send_byte((length >> 8) & 0xFF)
    _send_byte(length & 0xFF)

    # config data bytes, with 300 ms pause after first 20 bytes
    for i, b in enumerate(config_data):
        _send_byte(b)
        if i == 19:
            time.sleep(0.3)

    # RBCP ACK only confirms receipt. Wait for the final SPI word to finish.
    time.sleep(0.05)

    print(f"Si5345 config sent: {length} bytes to stage1={stage1_addr},"
          f" underwater={underwater_addr}")
    return length

def read_si5345(page, addr, stage1_addr, underwater_addr,
                udp_ip='192.168.31.16', udp_port=4660):
    """
    Read one Si5345 register via the 0xF1 slow-control command.

    Sends the 4-word read sequence, then issues exactly 2 collect reads
    (0xC0) to retrieve one 16-bit result from stage2 fifo_sc_tx.

    Parameters
    ----------
    page : str or int
        Si5345 page number, e.g. 0 or '00'.
    addr : str or int
        Si5345 register address within the page (low 8 bits),
        e.g. 0x2B or '2B'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.

    Returns
    -------
    result : str
        Hex string of the register value (2 chars, e.g. '08').
    """
    page_val = int(page, 16) if isinstance(page, str) else page
    addr_val = int(addr, 16) if isinstance(addr, str) else addr

    # Remove any response left by an aborted or previous transaction.
    udp_flush_slow_control(
        stage1_addr, udp_ip=udp_ip, udp_port=udp_port
    )

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF1 command
    _send_byte(0xF1)

    # 4 × 16-bit SPI words (8 bytes total)
    _send_byte(0x00);  _send_byte(0x01)   # word1: page register addr
    _send_byte(0x40);  _send_byte(page_val) # word2: write page
    _send_byte(0x00);  _send_byte(addr_val) # word3: register addr
    _send_byte(0x80);  _send_byte(0x00)     # word4: read command

    # The asymmetric 16-to-8 FIFO emits the high byte first.
    value = udp_read_word(
        stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port
    )

    result = f"{value & 0xFF:02X}"
    print(f"Si5345 read stage1={stage1_addr} uw={underwater_addr} "
          f"page=0x{page_val:02X} addr=0x{addr_val:02X} => 0x{value:04X}")
    return result

SI5345_API_VERSION = 3

### Si5345 按配置文件写入测试

解析 125 MHz 配置文件并通过慢控链写入 Si5345。

In [292]:
si5345_config_data_125M = parse_si5345_txt_file("Si5345-RevD-5345CONF-Registers_125M.txt")
config_si5345(si5345_config_data_125M, 1, 1)

Si5345 config sent: 2176 bytes to stage1=1, underwater=1


2176

### Si5345 全量寄存器回读比对测试

按配置文件顺序读取全部寄存器，区分真实不一致、控制寄存器预期差异和读错误。

In [325]:
import contextlib
import io
import time

SI5345_VERIFY_FILE = 'Si5345-RevD-5345CONF-Registers_125M.txt'
SI5345_VERIFY_STAGE1_ADDR = 1
SI5345_VERIFY_UNDERWATER_ADDR = 1
SI5345_VERIFY_RETRY_MISMATCH = True
SI5345_VERIFY_PROGRESS_EVERY = 25

# These addresses contain commands/trigger bits whose readback is not
# necessarily equal to the value that was written.
SI5345_CONTROL_REGISTERS = {
    0x0540, 0x0514, 0x001C, 0x0B24, 0x0B25,
}

# These preamble commands are intentionally overwritten or self-clear
# later in the ClockBuilder sequence.  Their final readback value is
# therefore different from the value written at the listed occurrence.
SI5345_EXPECTED_CONTROL_READBACK = {
    (0x0B24, 'preamble'): 0xC3,
    (0x0B25, 'preamble'): 0x02,
    (0x0540, 'preamble'): 0x00,
    (0x0514, 'postamble'): 0x00,
    (0x001C, 'postamble'): 0x00,
}


def parse_si5345_verify_file(file_path):
    records = []
    section = 'unknown'

    with open(file_path, 'r') as f:
        for line_no, raw_line in enumerate(f, 1):
            line = raw_line.strip()
            if 'Start configuration preamble' in line:
                section = 'preamble'
                continue
            if 'Start configuration registers' in line:
                section = 'configuration'
                continue
            if 'Start configuration postamble' in line:
                section = 'postamble'
                continue
            if not line or line.startswith('#') or ',' not in line:
                continue

            address_text, data_text = line.split(',', 1)
            try:
                address = int(address_text.strip(), 16)
                expected = int(data_text.strip(), 16)
            except ValueError:
                continue

            if not (0 <= address <= 0xFFFF and 0 <= expected <= 0xFF):
                raise ValueError(
                    f'Invalid register line {line_no}: {line}'
                )

            records.append({
                'line': line_no,
                'section': section,
                'address': address,
                'page': (address >> 8) & 0xFF,
                'register': address & 0xFF,
                'expected': expected,
            })
    return records


def read_si5345_quiet(page, register):
    # read_si5345() prints every access; suppress that inside the sweep.
    with contextlib.redirect_stdout(io.StringIO()):
        result = read_si5345(
            page, register,
            SI5345_VERIFY_STAGE1_ADDR,
            SI5345_VERIFY_UNDERWATER_ADDR,
        )
    return int(result, 16)


records = parse_si5345_verify_file(SI5345_VERIFY_FILE)
si5345_readback_rows = []
si5345_readback_mismatches = []
si5345_readback_errors = []
si5345_readback_recovered = []
si5345_readback_expected_control = []

print(
    f'Checking {len(records)} Si5345 register writes from '
    f'{SI5345_VERIFY_FILE}'
)

for index, record in enumerate(records, 1):
    page = record['page']
    register = record['register']
    address = record['address']
    expected = record['expected']

    try:
        actual = read_si5345_quiet(page, register)
        status = 'match' if actual == expected else 'mismatch'

        if status == 'mismatch' and SI5345_VERIFY_RETRY_MISMATCH:
            time.sleep(0.01)
            actual_retry = read_si5345_quiet(page, register)
            if actual_retry == expected:
                si5345_readback_recovered.append({
                    **record,
                    'first_actual': actual,
                    'actual': actual_retry,
                })
                actual = actual_retry
                status = 'match-after-retry'
            else:
                actual = actual_retry
                status = 'mismatch'

        control_key = (address, record['section'])
        expected_final = SI5345_EXPECTED_CONTROL_READBACK.get(control_key)
        if status == 'mismatch' and actual == expected_final:
            status = 'expected-control-readback'

        row = {
            **record,
            'index': index,
            'actual': actual,
            'status': status,
        }
        si5345_readback_rows.append(row)
        if status == 'mismatch':
            si5345_readback_mismatches.append(row)
        elif status == 'expected-control-readback':
            si5345_readback_expected_control.append(row)

    except Exception as exc:
        error = {
            **record,
            'index': index,
            'error': str(exc),
        }
        si5345_readback_errors.append(error)

    if index % SI5345_VERIFY_PROGRESS_EVERY == 0 or index == len(records):
        print(
            f'progress {index}/{len(records)}, '
            f'mismatches={len(si5345_readback_mismatches)}, '
            f'errors={len(si5345_readback_errors)}'
        )

matched = len(si5345_readback_rows) - len(si5345_readback_mismatches)

print()
print('=' * 78)
print(
    f'Si5345 readback summary: checked={len(si5345_readback_rows)}, '
    f'matched={matched}, mismatched={len(si5345_readback_mismatches)}, '
    f'expected-control={len(si5345_readback_expected_control)}, '
    f'errors={len(si5345_readback_errors)}, '
    f'recovered-by-retry={len(si5345_readback_recovered)}'
)

if si5345_readback_expected_control:
    print('Expected control-register readback differences:')
    for row in si5345_readback_expected_control:
        print(
            f"0x{row['address']:04X} ({row['section']}): "
            f"written=0x{row['expected']:02X}, "
            f"final-readback=0x{row['actual']:02X}"
        )

if si5345_readback_mismatches:
    print('Mismatched registers (file order):')
    for row in si5345_readback_mismatches:
        special = ' [control-register]' if row['address'] in SI5345_CONTROL_REGISTERS else ''
        print(
            f"[{row['index']:3d}] line={row['line']:3d} "
            f"{row['section']:13s} 0x{row['address']:04X} "
            f"(page=0x{row['page']:02X}, reg=0x{row['register']:02X}) "
            f"expected=0x{row['expected']:02X} "
            f"actual=0x{row['actual']:02X}{special}"
        )
else:
    print('All readable registers match the configuration file.')

if si5345_readback_recovered:
    print('Transient mismatches recovered by the second read:')
    for row in si5345_readback_recovered:
        print(
            f"0x{row['address']:04X}: "
            f"first=0x{row['first_actual']:02X}, "
            f"retry=0x{row['actual']:02X}, "
            f"expected=0x{row['expected']:02X}"
        )

if si5345_readback_errors:
    print('Read errors:')
    for row in si5345_readback_errors:
        print(
            f"[{row['index']:3d}] line={row['line']:3d} "
            f"0x{row['address']:04X}: {row['error']}"
        )

si5345_verify_result = {
    'records': records,
    'rows': si5345_readback_rows,
    'mismatches': si5345_readback_mismatches,
    'errors': si5345_readback_errors,
    'recovered': si5345_readback_recovered,
    'expected_control': si5345_readback_expected_control,
}

Checking 526 Si5345 register writes from Si5345-RevD-5345CONF-Registers_125M.txt
progress 25/526, mismatches=0, errors=0
progress 50/526, mismatches=0, errors=0
progress 75/526, mismatches=0, errors=0
progress 100/526, mismatches=0, errors=0
progress 125/526, mismatches=0, errors=0
progress 150/526, mismatches=0, errors=0
progress 175/526, mismatches=0, errors=0
progress 200/526, mismatches=0, errors=0
progress 225/526, mismatches=0, errors=0
progress 250/526, mismatches=0, errors=0
progress 275/526, mismatches=0, errors=0
progress 300/526, mismatches=0, errors=0
progress 325/526, mismatches=0, errors=0
progress 350/526, mismatches=0, errors=0
progress 375/526, mismatches=0, errors=0
progress 400/526, mismatches=0, errors=0
progress 425/526, mismatches=0, errors=0
progress 450/526, mismatches=0, errors=0
progress 475/526, mismatches=0, errors=0
progress 500/526, mismatches=0, errors=0
progress 525/526, mismatches=0, errors=0
progress 526/526, mismatches=0, errors=0

Si5345 readback sum

### Si5345 单寄存器读取测试

检查函数 API 版本并读取一个指定寄存器。

In [294]:
if (globals().get('DAQ_RBCP_API_VERSION') != 4 or
        globals().get('SI5345_API_VERSION') != 3):
    raise RuntimeError(
        'Old notebook functions are loaded. Restart Kernel and Run All.'
    )

print("Si5345 0x0051 =", read_si5345(0x01, 0x1C, 1, 1))

Si5345 read stage1=1 uw=1 page=0x01 addr=0x1C => 0x0006
Si5345 0x0051 = 06


## 4. AD9253 ADC 配置、Bitslip 与 IDELAY

### AD9253 配置、写入和单寄存器读取函数

包含十六进制参数转换、ADC mixed-frequency 配置和单寄存器回读。只定义函数。

In [299]:
def str_to_byte(value_str):
    """
    Convert a value string into one byte.

    Supports:
        '0xFF'
        'FF'
        '11111111'
        '255'
    Returns:
        b'\xFF'
    """
    value_str = value_str.strip()

    # binary
    if len(value_str) == 8 and set(value_str) <= {"0", "1"}:
        value = int(value_str, 2)
    # hex, with 0x prefix
    elif value_str.lower().startswith("0x"):
        value = int(value_str, 16)
    # hex, no 0x prefix
    elif len(value_str) == 2:
        value = int(value_str, 16)
    # decimal
    else:
        value = int(value_str)

    if not (0 <= value <= 0xFF):
        raise ValueError("Value must be between 0 and 255.")

    return bytes([value])


def config_ad9253(adc_num, test_mode, stage1_addr, underwater_addr,
                  udp_ip='192.168.31.16', udp_port=4660):
    """
    Configure one AD9253 ADC via the 0xF2 bulk-write slow-control command.

    Each byte is sent as a separate UDP write packet (matching the 1-byte
    FIFO word push in stage2 rbcp_slow_control).

    AD9253 SPI words are 3 bytes: [reg_hi, reg_lo, data].

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    test_mode : str
        "none", "1bit high", "checkerboard", "full scale" or
        "mixed frequency".
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """
    # 14bit, 0x56=125M, 0x50=20M
    sample_rate_reg = b'\x01\x00\x56'
    # write 0x01 into 0xFF after touching 0x100-region registers
    confirm_reg = b'\x00\xFF\x01'

    # test mode register 0x0D:
    #   0x0B: 1bit high, 0x04: checkerboard, 0x02: full scale,
    #   0x0C: mixed frequency
    if test_mode == "none":
        test_mode_reg = b'\x00\x0D\x00'
    elif test_mode == "1bit high":
        test_mode_reg = b'\x00\x0D\x0B'
    elif test_mode == "checkerboard":
        test_mode_reg = b'\x00\x0D\x04'
    elif test_mode == "full scale":
        test_mode_reg = b'\x00\x0D\x02'
    elif test_mode == "mixed frequency":
        test_mode_reg = b'\x00\x0D\x0C'
    else:
        raise ValueError(f"Invalid test_mode: {test_mode}")

    # output mode, 0x00 = offset binary
    output_mode_reg = b'\x00\x14\x00'

    config_data = (sample_rate_reg + confirm_reg + test_mode_reg +
                   output_mode_reg)
    length = len(config_data)

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF2 command byte
    _send_byte(0xF2)

    # ADC number
    _send_byte(adc_num)

    # 2-byte big-endian length
    _send_byte((length >> 8) & 0xFF)
    _send_byte(length & 0xFF)

    # config data bytes
    for b in config_data:
        _send_byte(b)

    time.sleep(0.02)
    print(f"AD9253 config sent: {length} bytes to stage1={stage1_addr},"
          f" underwater={underwater_addr}, adc={adc_num}, "
          f"mode={test_mode}")


def usr_test_ad9253(adc_num, D1_M, D1_L, D2_M, D2_L, stage1_addr,
                    underwater_addr, udp_ip='192.168.31.16',
                    udp_port=4660):
    """
    Write AD9253 channel gain/offset registers via the 0xF2 command.

    Configures D1/D2 midscale and lowscale offsets plus test mode,
    used for ADC gain calibration.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    D1_M, D1_L, D2_M, D2_L : str
        Midscale / lowscale offset values for channel 1 and 2.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """
    sample_rate_reg = b'\x01\x00\x56'
    confirm_reg = b'\x00\xFF\x01'

    D1_M_reg = b'\x00\x1A' + str_to_byte(D1_M)
    D1_L_reg = b'\x00\x19' + str_to_byte(D1_L)
    D2_M_reg = b'\x00\x1C' + str_to_byte(D2_M)
    D2_L_reg = b'\x00\x1B' + str_to_byte(D2_L)
    test_mode_reg = b'\x00\x0D\x08'

    # output mode, 0x00 = offset binary
    output_mode_reg = b'\x00\x14\x00'

    config_data = (sample_rate_reg + confirm_reg + D1_M_reg + D1_L_reg +
                   D2_M_reg + D2_L_reg + test_mode_reg + output_mode_reg)
    length = len(config_data)

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF2 command byte
    _send_byte(0xF2)

    # ADC number
    _send_byte(adc_num)

    # 2-byte big-endian length
    _send_byte((length >> 8) & 0xFF)
    _send_byte(length & 0xFF)

    # config data bytes
    for b in config_data:
        _send_byte(b)

    time.sleep(0.02)
    print(f"AD9253 user-test config sent: {length} bytes to "
          f"stage1={stage1_addr}, underwater={underwater_addr}, "
          f"adc={adc_num}")


def read_ad9253(adc_num, addr1, addr2, stage1_addr, underwater_addr,
                udp_ip='192.168.31.16', udp_port=4660):
    """
    Read one AD9253 register via the 0xF3 slow-control command.

    Sends the 0xF3 command, ADC number and the read address (with bit7 of
    addr1 set), then issues exactly 2 collect reads (0xC0) to retrieve one
    16-bit result from stage2 fifo_sc_tx.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    addr1, addr2 : str
        High / low byte of the AD9253 register address, e.g. '0x00', '0x0D'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.

    Returns
    -------
    result : str
        Hex string of the register value (2 chars, e.g. '0B').
    """
    # Remove any response left by an aborted or previous transaction.
    udp_flush_slow_control(
        stage1_addr, udp_ip=udp_ip, udp_port=udp_port
    )

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF3 command byte
    _send_byte(0xF3)

    # ADC number
    _send_byte(adc_num)

    # read address (bit7 of first byte = read flag)
    _send_byte(int(addr1, 16) | 0x80)
    _send_byte(int(addr2, 16))

    value = udp_read_word(
        stage1_addr, underwater_addr,
        udp_ip=udp_ip, udp_port=udp_port
    )
    result = f"{value & 0xFF:02X}"
    print(f"AD9253 read stage1={stage1_addr} uw={underwater_addr} "
          f"adc={adc_num} addr={addr1}|{addr2} => 0x{result}")
    return result


def configure_all_ad9253(mode, stage1_addr=1, underwater_addr=1,
                         max_attempts=3):
    """Configure all eight ADCs and verify register 0x0D."""
    expected_modes = {
        "mixed frequency": "0C",
        "none": "00",
    }
    if mode not in expected_modes:
        raise ValueError(f"Unsupported ADC mode: {mode}")

    expected = expected_modes[mode]
    readback = {}
    for adc_num in range(8):
        value = None
        for attempt in range(max_attempts):
            config_ad9253(
                adc_num, mode, stage1_addr, underwater_addr
            )
            time.sleep(0.1)
            value = read_ad9253(
                adc_num, "00", "0D", stage1_addr, underwater_addr
            )
            if value == expected:
                break
            print(
                f"ADC{adc_num} {mode} readback {value} != {expected}, "
                f"retry {attempt + 1}/{max_attempts}"
            )
            time.sleep(0.05)
        readback[adc_num] = value
    return readback


### AD9253 Bitslip 函数

根据 mixed-frequency 目标码型执行单通道 Bitslip。只定义函数。

In [300]:
def ad9253_bitslip(adc_num, channel, stage1_addr, underwater_addr,
                       udp_ip='192.168.31.16', udp_port=4660,
                       time_err=None, retry_delay=0.02,
                       max_attempts=16, verbose=True):
    """
    Align one AD9253 channel output via the 0xF4 bitslip slow-control
    command.

    In UDP mode each byte is sent as a separate 1-byte write packet
    (the same 1-byte FIFO word push a UDP packet does, same mechanism
    as every other slow-control byte). After the FPGA performs one bit
    slip it pushes the captured channel pattern as a 16-bit word
    {8'b0, value} into fifo_sc_tx, which is popped with 2 collect reads
    (0xC0); the low byte is compared against the target pattern.

    The loop keeps issuing bitslips until the captured pattern matches
    the channel's target, or the timeout expires.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    channel : str
        Channel name: 'cha0','cha1','chb0','chb1','chc0','chc1',
        'chd0','chd1'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    time_err : float
        Timeout for the whole bitslip loop in seconds.

    Returns
    -------
    bool
        True if the channel aligned to the target pattern, False on
        timeout.
    """

    # 通道 → 指令映射
    channel_map = {
        "cha0": 0,
        "cha1": 1,
        "chb0": 2,
        "chb1": 3,
        "chc0": 4,
        "chc1": 5,
        "chd0": 6,
        "chd1": 7,
    }

    # 通道 → 正确数据模式映射
    target_map = {
        "cha0": "10011100",
        "cha1": "10100001",
        "chb0": "10011100",
        "chb1": "10100001",
        "chc0": "10011100",
        "chc1": "10100001",
        "chd0": "10011100",
        "chd1": "10100001",
    }
    
    if not isinstance(adc_num, int) or not (0 <= adc_num <= 7):
        raise ValueError("adc_num must be an integer between 0 and 7")

    if channel not in channel_map:
        raise ValueError(f"Invalid channel: {channel}")

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    def _read_word():
        return udp_read_word(
            stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    channel_num = adc_num * 8 + channel_map[channel]

    start_time = time.time()

    for _attempt in range(max_attempts):
        if time_err is not None and time.time() - start_time > time_err:
            break

        # 0xF4 bitslip command
        _send_byte(0xF4)
        _send_byte(channel_num)

        time.sleep(0.01)

        data = _read_word()
        result = format(data & 0xFF, '08b')
        if verbose:
            print(
                f"ADC{adc_num} {channel} bitslip "
                f"attempt {_attempt + 1}: 0x{data & 0xFF:02X} {result}"
            )

        if result == target_map[channel]:
            if verbose:
                print(f"ADC{adc_num} {channel} bitslip done.")
            return True

        time.sleep(retry_delay)

    if verbose:
        print(f"ADC{adc_num} {channel} bitslip error after {max_attempts} attempts.")
    return False

### AD9253 IDELAY 与 delay_cali 函数

包含 IDELAY tap 设置、最佳窗口搜索和完整标定流程。只定义函数。

In [301]:
def idelay(adc_num, channel, delay_tap,
                       stage1_addr, underwater_addr,
                       udp_ip='192.168.31.16', udp_port=4660):
    """
    Set one AD9253 channel's input IDELAY tap via the 0xF5 slow-control
    command (single-tap write, no readback).

    In UDP mode each command byte is sent as a separate 1-byte write
    packet (matching the 1-byte FIFO word push in the slow-control
    manager, same as every other slow-control byte). This writes the
    three bytes 0xF5 / channel_num / delay_tap and reads nothing back.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7 on the underwater board.
    channel : str
        Channel name, one of 'cha0','cha1','chb0','chb1','chc0','chc1',
        'chd0','chd1'.
    delay_tap : int
        IDELAY tap value, 0~31.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    """

    channel_map = {
        "cha0": 0, "cha1": 1,
        "chb0": 2, "chb1": 3,
        "chc0": 4, "chc1": 5,
        "chd0": 6, "chd1": 7,
    }

    if not isinstance(adc_num, int) or not (0 <= adc_num <= 7):
        raise ValueError("adc_num must be an integer between 0 and 7")

    if channel not in channel_map:
        raise ValueError(f"Invalid channel: {channel}")

    if not isinstance(delay_tap, int) or not (0 <= delay_tap <= 31):
        raise ValueError("delay_tap must be an integer between 0 and 31")

    channel_num = adc_num * 8 + channel_map[channel]

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    _send_byte(0xF5)
    _send_byte(channel_num)
    _send_byte(delay_tap)


def find_best_delays(delays_matrix):
    """
    Pick the best IDELAY tap per channel from a 32-tap eye/valid matrix.

    Handles the circular sweep: appends a copy of each row so a valid run
    that wraps 31→0 is treated as one contiguous segment. The chosen tap
    is the midpoint of the longest valid run, mapped back to 0~31.

    Parameters
    ----------
    delays_matrix : numpy.ndarray
        Shape (n_channels, 32) of 0/1 validity flags per tap.

    Returns
    -------
    list
        Best tap (int 0~31) per channel, or None for a channel with no
        valid tap.
    """
    import numpy as np

    delays = []

    for ch in range(delays_matrix.shape[0]):
        row = delays_matrix[ch]
        n = len(row)

        # 拼接一份，处理环形
        extended = np.concatenate([row, row])

        max_len = 0
        max_start = 0

        curr_len = 0
        curr_start = 0

        for i in range(len(extended)):
            if extended[i] == 1:
                if curr_len == 0:
                    curr_start = i
                curr_len += 1

                if curr_len > max_len:
                    max_len = curr_len
                    max_start = curr_start
            else:
                curr_len = 0

        if max_len == 0:
            # 没有可用点
            delays.append(None)
            continue

        # 中点（取整数）
        center = max_start + max_len // 2

        # 映射回 0~31
        delay = center % n

        delays.append(delay)

    return delays


def ad9253_delay_cali(adc_num, stage1_addr, underwater_addr,
                      udp_ip='192.168.31.16', udp_port=4660,
                      time_err=None, settle_delay=0.002,
                      apply_best=True, sweep_max_attempts=12,
                      final_max_attempts=24, verify_rounds=2,
                      verbose=False):
    """
    Full 8-channel x 32-tap IDELAY calibration for one AD9253 ADC.

    For every (channel, delay_tap) it writes the tap via the 0xF5
    command and runs ad9253_bitslip (0xF4) to check whether the
    channel aligns under that tap, filling a (1,32) validity row per
    channel. The 8x32 matrix is printed are_done and passed to
    find_best_delays for the per-channel best tap.

    Note: only the sweep and best-delay selection are performed here;
    re-applying the best taps (0xF5 + ad9253_bitslip) is left as
    the commented-out loop at the end so you can inspect delays_matrix
    first.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    time_err : float
        Per-bitslip timeout passed to ad9253_bitslip.
    """

    import numpy as np

    delays_matrix = np.zeros((8, 32), dtype=int)
    channel_map = ["cha0", "cha1", "chb0", "chb1",
                   "chc0", "chc1", "chd0", "chd1"]

    for channel in range(8):
        for delay_tap in range(32):
            idelay(adc_num, channel_map[channel], delay_tap,
                              stage1_addr, underwater_addr,
                              udp_ip=udp_ip, udp_port=udp_port)
            time.sleep(settle_delay)
            if ad9253_bitslip(adc_num, channel_map[channel],
                                  stage1_addr, underwater_addr,
                                  udp_ip=udp_ip, udp_port=udp_port,
                                  time_err=time_err,
                                  retry_delay=0.01,
                                  max_attempts=sweep_max_attempts,
                                  verbose=verbose):
                delays_matrix[channel, delay_tap] = 1
            time.sleep(settle_delay)

    best_delays = find_best_delays(delays_matrix)
    aligned = []

    if apply_best:
        for channel, delay_tap in enumerate(best_delays):
            if delay_tap is None:
                aligned.append(False)
                continue
            idelay(adc_num, channel_map[channel], delay_tap,
                   stage1_addr, underwater_addr,
                   udp_ip=udp_ip, udp_port=udp_port)
            time.sleep(settle_delay)
            ok = True
            for _ in range(verify_rounds):
                ok = ad9253_bitslip(
                    adc_num, channel_map[channel],
                    stage1_addr, underwater_addr,
                    udp_ip=udp_ip, udp_port=udp_port,
                    time_err=time_err,
                    retry_delay=0.02,
                    max_attempts=final_max_attempts,
                    verbose=verbose
                )
                if not ok:
                    break
            aligned.append(ok)

    if verbose:
        print(f"ADC{adc_num} delay matrix:\n{delays_matrix}")
    print(f"ADC{adc_num} best delays: {best_delays}")
    if apply_best:
        print(f"ADC{adc_num} final bitslip: {aligned}")

    return delays_matrix, best_delays, aligned


def ad9253_delay_cali_1ch(adc_num, channel, stage1_addr,
                              underwater_addr,
                              udp_ip='192.168.31.16', udp_port=4660,
                              time_err=1.0):
    """
    Single-channel 32-tap IDELAY calibration for one AD9253 channel.

    Prints the (1,32) validity row for the given channel; the tap that
    aligns (matched by ad9253_bitslip) is marked 1. Use
    find_best_delays on this row to pick the window midpoint tap.

    Parameters
    ----------
    adc_num : int
        ADC index 0~7.
    channel : str
        Channel name, e.g. 'cha0'.
    stage1_addr : int
        Stage1 board address, 1~8.
    underwater_addr : int
        Underwater board address, 1~8.
    udp_ip : str
        SiTCP target IP.
    udp_port : int
        SiTCP target UDP port.
    time_err : float
        Per-bitslip timeout passed to ad9253_bitslip.
    """

    import numpy as np

    delays_matrix = np.zeros((1, 32), dtype=int)

    for delay_tap in range(32):
        idelay(adc_num, channel, delay_tap,
                          stage1_addr, underwater_addr,
                          udp_ip=udp_ip, udp_port=udp_port)
        time.sleep(0.05)
        if ad9253_bitslip(adc_num, channel,
                              stage1_addr, underwater_addr,
                              udp_ip=udp_ip, udp_port=udp_port,
                              time_err=time_err):
            delays_matrix[0, delay_tap] = 1
        time.sleep(0.05)

    print(delays_matrix)



### AD9253 单通道 IDELAY 扫描测试

对一个 ADC 的单通道扫描 32 个 tap。

In [271]:
ad9253_delay_cali_1ch(1, 'chd1', 1, 1)

ADC1 chd1 bitslip attempt 1: 0xA1 10100001
ADC1 chd1 bitslip done.
ADC1 chd1 bitslip attempt 1: 0x0D 00001101
ADC1 chd1 bitslip attempt 2: 0x86 10000110
ADC1 chd1 bitslip attempt 3: 0x34 00110100
ADC1 chd1 bitslip attempt 4: 0x1A 00011010
ADC1 chd1 bitslip attempt 5: 0xD0 11010000
ADC1 chd1 bitslip attempt 6: 0x68 01101000
ADC1 chd1 bitslip attempt 7: 0x43 01000011
ADC1 chd1 bitslip attempt 8: 0xA1 10100001
ADC1 chd1 bitslip done.
ADC1 chd1 bitslip attempt 1: 0x0D 00001101
ADC1 chd1 bitslip attempt 2: 0x86 10000110
ADC1 chd1 bitslip attempt 3: 0x34 00110100
ADC1 chd1 bitslip attempt 4: 0x1A 00011010
ADC1 chd1 bitslip attempt 5: 0xD0 11010000
ADC1 chd1 bitslip attempt 6: 0x68 01101000
ADC1 chd1 bitslip attempt 7: 0x43 01000011
ADC1 chd1 bitslip attempt 8: 0xA1 10100001
ADC1 chd1 bitslip done.
ADC1 chd1 bitslip attempt 1: 0x0D 00001101
ADC1 chd1 bitslip attempt 2: 0x86 10000110
ADC1 chd1 bitslip attempt 3: 0x34 00110100
ADC1 chd1 bitslip attempt 4: 0x1A 00011010
ADC1 chd1 bitslip attempt

### AD9253 配置为 mixed-frequency test mode

配置全部 ADC，并回读 0x0D；只有读到 0C 的 ADC 才会参加后续标定。

In [302]:
# Put every ADC into mixed-frequency test mode.
ADC_STAGE1_ADDR = 1
ADC_UNDERWATER_ADDR = 1

adc_mode_readback = configure_all_ad9253(
    "mixed frequency",
    stage1_addr=ADC_STAGE1_ADDR,
    underwater_addr=ADC_UNDERWATER_ADDR,
    max_attempts=3,
)
print("ADC test-mode readback:", adc_mode_readback)

configured_adcs = [
    adc for adc, value in adc_mode_readback.items() if value == "0C"
]
failed_adcs = [
    adc for adc, value in adc_mode_readback.items() if value != "0C"
]
print("ADCs ready for calibration:", configured_adcs)
if failed_adcs:
    print("ADC configuration failed for:", failed_adcs)


AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=0, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=0 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=1, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=1 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=2, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=2 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=3, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=3 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=4, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=4 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=5, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=5 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=6, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=6 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1,

### AD9253 独立 delay_cali 标定

前提：先运行 test-mode 配置块，并确保全部 ADC 的 0x0D=0C。本块只扫描并保存最佳 delay，不应用，也不切换回 normal mode。

In [ ]:
# Calibrate ADC delay windows only.  Run the test-mode configuration
# block first.  Results are stored in adc_calibration_results and, when
# complete, persisted to adc_delay_values.json.
import json

ADC_CALI_VERBOSE = False
ADC_SWEEP_ATTEMPTS = 12
ADC_FINAL_ATTEMPTS = 24
ADC_VERIFY_ROUNDS = 2

channel_names = [
    "cha0", "cha1", "chb0", "chb1",
    "chc0", "chc1", "chd0", "chd1",
]

calibration_modes = globals().get("adc_mode_readback", {})
configured_adcs = [
    adc for adc in range(8)
    if calibration_modes.get(adc) == "0C"
]
if not configured_adcs:
    raise RuntimeError(
        "No ADC is in mixed-frequency mode. Run the ADC test-mode "
        "configuration block first."
    )

adc_calibration_results = {}

for adc_num in configured_adcs:
    matrix, best_delays, _ = ad9253_delay_cali(
        adc_num,
        ADC_STAGE1_ADDR,
        ADC_UNDERWATER_ADDR,
        apply_best=False,
        sweep_max_attempts=ADC_SWEEP_ATTEMPTS,
        final_max_attempts=ADC_FINAL_ATTEMPTS,
        verify_rounds=ADC_VERIFY_ROUNDS,
        verbose=ADC_CALI_VERBOSE,
    )

    adc_calibration_results[adc_num] = {
        "matrix": matrix,
        "best_delays": best_delays,
    }

    print(f"ADC{adc_num} best delays:", dict(zip(channel_names, best_delays)))
    print("delay matrix: 1=valid, 0=invalid")
    print("tap   " + " ".join(f"{tap:2d}" for tap in range(32)))
    for channel, row in zip(channel_names, matrix):
        values = " ".join(f"{int(value):2d}" for value in row)
        print(f"{channel:5s} {values}")

best_delay_matrix = []
for adc_num in range(8):
    result = adc_calibration_results.get(adc_num)
    if result is None:
        best_delay_matrix.append(None)
        continue
    best_delays = result["best_delays"]
    if len(best_delays) != 8 or any(value is None for value in best_delays):
        best_delay_matrix.append(None)
        continue
    best_delay_matrix.append([int(value) for value in best_delays])

if all(values is not None for values in best_delay_matrix):
    adc_delay_value = best_delay_matrix
    with open("adc_delay_values.json", "w", encoding="utf-8") as f:
        json.dump(adc_delay_value, f, indent=2)
        f.write("\n")
    print("Calibration complete. Updated adc_delay_value and saved adc_delay_values.json.")
else:
    print(
        "Calibration is incomplete. The existing adc_delay_value matrix "
        "was not overwritten."
    )


### AD9253 已保存的 delay 结果

这是持久保存在 notebook 中的 8×8 delay 矩阵，没有被删除。标定完成后也会同步写入 `adc_delay_values.json`。

In [303]:
adc0_delay_value = [22, 21, 10, 22, 10, 10, 10, 20]
adc1_delay_value = [11, 9, 24, 10, 10, 10, 11, 11]
adc2_delay_value = [5, 5, 5, 18, 5, 18, 5, 7]
adc3_delay_value = [19, 8, 20, 21, 19, 9, 19, 23]
adc4_delay_value = [13, 25, 13, 25, 13, 24, 13, 25]
adc5_delay_value = [13, 12, 12, 12, 25, 12, 12, 12]
adc6_delay_value = [24, 23, 24, 21, 10, 24, 10, 12]
adc7_delay_value = [8, 9, 8, 8, 8, 8, 0, 7]
adc_delay_value = [adc0_delay_value, adc1_delay_value, adc2_delay_value, adc3_delay_value, adc4_delay_value, adc5_delay_value, adc6_delay_value, adc7_delay_value]

### AD9253 应用已存 delay 并执行 Bitslip

先把全部 ADC 配置为 mixed-frequency test mode，再应用上面的 delay 矩阵并执行 Bitslip，最后强制把全部 ADC 配置回 normal mode。

In [304]:
# Apply the saved delay matrix, run Bitslip in mixed-frequency mode,
# then unconditionally restore every ADC to normal mode.
import json
import os

STAGE1_ADDR = 1
UNDERWATER_ADDR = 1
BITSLIP_ATTEMPTS = 24
BITSLIP_ROUNDS = 2

channel_names = [
    "cha0", "cha1", "chb0", "chb1",
    "chc0", "chc1", "chd0", "chd1",
]

stored_delay_value = globals().get("adc_delay_value")
if stored_delay_value is None and os.path.exists("adc_delay_values.json"):
    with open("adc_delay_values.json", "r", encoding="utf-8") as f:
        stored_delay_value = json.load(f)
if stored_delay_value is None:
    raise RuntimeError(
        "No stored ADC delay matrix. Run the stored-delay block or "
        "the ADC calibration block first."
    )

if len(stored_delay_value) != 8 or any(
    len(values) != 8 for values in stored_delay_value
):
    raise ValueError("stored delay matrix must be 8 x 8")
for adc_num, values in enumerate(stored_delay_value):
    for channel_num, value in enumerate(values):
        if not isinstance(value, int) or not (0 <= value <= 31):
            raise ValueError(
                f"Invalid delay ADC{adc_num} channel {channel_num}: {value}"
            )

adc_delay_apply_results = {}
normal_mode_readback = {}

try:
    test_mode_readback = configure_all_ad9253(
        "mixed frequency",
        stage1_addr=STAGE1_ADDR,
        underwater_addr=UNDERWATER_ADDR,
        max_attempts=3,
    )
    print("ADC test-mode readback:", test_mode_readback)

    configured_adcs = [
        adc for adc, value in test_mode_readback.items() if value == "0C"
    ]

    for adc_num in configured_adcs:
        delays = stored_delay_value[adc_num]
        aligned = []
        print(f"\n=== ADC{adc_num} ===")

        for channel, delay_tap in zip(channel_names, delays):
            idelay(
                adc_num, channel, delay_tap,
                STAGE1_ADDR, UNDERWATER_ADDR,
            )
            time.sleep(0.01)

            ok = True
            for _round in range(BITSLIP_ROUNDS):
                ok = ad9253_bitslip(
                    adc_num,
                    channel,
                    STAGE1_ADDR,
                    UNDERWATER_ADDR,
                    retry_delay=0.02,
                    max_attempts=BITSLIP_ATTEMPTS,
                    verbose=False,
                )
                if not ok:
                    break

            aligned.append(ok)
            print(
                f"ADC{adc_num} {channel}: delay={delay_tap:2d}, "
                f"aligned={ok}"
            )

        adc_delay_apply_results[adc_num] = {
            "delays": list(delays),
            "aligned": aligned,
        }

    for adc_num in range(8):
        if adc_num not in adc_delay_apply_results:
            adc_delay_apply_results[adc_num] = {
                "delays": list(stored_delay_value[adc_num]),
                "aligned": [False] * 8,
            }
finally:
    normal_mode_readback = configure_all_ad9253(
        "none",
        stage1_addr=STAGE1_ADDR,
        underwater_addr=UNDERWATER_ADDR,
        max_attempts=3,
    )
    print("ADC normal-mode readback:", normal_mode_readback)

print("\n=== Stored-delay apply summary ===")
for adc_num, result in adc_delay_apply_results.items():
    failed = [
        channel
        for channel, ok in zip(channel_names, result["aligned"])
        if not ok
    ]
    print(
        f"ADC{adc_num}: delays={result['delays']}, "
        f"failed={failed if failed else 'none'}"
    )


AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=0, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=0 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=1, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=1 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=2, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=2 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=3, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=3 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=4, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=4 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=5, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=5 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1, adc=6, mode=mixed frequency
AD9253 read stage1=1 uw=1 adc=6 addr=00|0D => 0x0C
AD9253 config sent: 12 bytes to stage1=1, underwater=1,

## 5. DAC128S085

### DAC128S085 配置函数

按通道电压计算 DAC 寄存器值并通过慢控链配置。只定义函数。

In [305]:
def config_dac128s085(dac_num, channel, voltage,
                        stage1_addr, underwater_addr,
                        udp_ip='192.168.31.16', udp_port=4660):
    """
    配置 DAC128S085。

    在 UDP 模式下每个命令字节被当作独立的 1 字节写包发送（与其它慢控
    字节的 1 字节 FIFO 压栈机制一致）。写入 0xF6 / dac_num / WTM 模式
    字节，再写入 0xF6 / dac_num / 16 位大端数据。无读回。

    Parameters
    ----------
    dac_num : int
        DAC 芯片号，0~7。
    channel : str
        通道名，'A'/'B'/'C'/'D'/'ALL'。
    voltage : float
        输出电压，0~3.3 V。
    stage1_addr : int
        Stage1 板地址，1~8。
    underwater_addr : int
        水下板地址，1~8。
    udp_ip : str
        SiTCP 目标 IP。
    udp_port : int
        SiTCP 目标 UDP 端口。
    """

    if not isinstance(dac_num, int) or not (0 <= dac_num <= 7):
        raise ValueError("dac_num must be an integer between 0 and 7")

    if voltage < 0 or voltage > 3.3:
        raise ValueError("电压值必须在 0~3.3 V 之间")

    # 12 位 DAC 码
    D = int(voltage * 4096 / 3.3)

    # CMD 决定通道
    if channel.upper() == "A":
        cmd = 0b0011
    elif channel.upper() == "B":
        cmd = 0b0010
    elif channel.upper() == "C":
        cmd = 0b0001
    elif channel.upper() == "D":
        cmd = 0b0000
    elif channel.upper() == "ALL":
        cmd = 0b1100
    else:
        raise ValueError("不支持的通道，有效通道为: A, B, C, D, ALL")

    # DB[15:12] = cmd, DB[11:0] = D
    db = (cmd << 12) | (D & 0xFFF)

    def _send_byte(b):
        udp_write_byte(
            b, stage1_addr, underwater_addr,
            udp_ip=udp_ip, udp_port=udp_port
        )

    # 0xF6 命令
    _send_byte(0xF6)
    _send_byte(dac_num)

    # WTM 模式配置
    _send_byte(0b10010000)
    _send_byte(0b00000000)

    # 写入 16 位数据（大端）
    _send_byte(0xF6)
    _send_byte(dac_num)
    _send_byte((db >> 8) & 0xFF)
    _send_byte(db & 0xFF)


def trig_random_2ch(comp_input_static,
                     stage1_addr, underwater_addr,
                     udp_ip='192.168.31.16', udp_port=4660):
    """
    随机选中两个通道并写入比较器静态输入电压。

    在 0~31 中随机取两个不重复整数，映射为 (dac_num, channel)，
    再调用 config_dac128s085 写入 comp_input_static 对应值
    （加 0.015 V 偏移）。先对所有 8 块 DAC 的 ALL 通道写 2.1 V。

    Parameters
    ----------
    comp_input_static : list[list[float]]
        8×4 的比较器静态输入电压矩阵。
    stage1_addr : int
        Stage1 板地址，1~8。
    underwater_addr : int
        水下板地址，1~8。
    udp_ip : str
        SiTCP 目标 IP。
    udp_port : int
        SiTCP 目标 UDP 端口。
    """

    import random

    # 随机生成 0~31
    num1, num2 = random.sample(range(32), 2)

    # 计算 dac_num 和 channel
    dac_num1 = num1 // 4
    channel1 = num1 % 4
    dac_num2 = num2 // 4
    channel2 = num2 % 4

    print(f"随机数1: {num1}")
    print(f"dac_num1: {dac_num1}")
    print(f"channel1: {channel1}")
    print(f"随机数2: {num2}")
    print(f"dac_num2: {dac_num2}")
    print(f"channel2: {channel2}")

    # 先将所有 DAC 的 ALL 通道配置为 2.1 V
    for i in range(8):
        config_dac128s085(i, "ALL", 2.1,
                            stage1_addr, underwater_addr,
                            udp_ip=udp_ip, udp_port=udp_port)

    config_dac128s085(dac_num1, channel1,
                       comp_input_static[dac_num1][channel1] + 0.015,
                       stage1_addr, underwater_addr,
                       udp_ip=udp_ip, udp_port=udp_port)
    config_dac128s085(dac_num2, channel2,
                       comp_input_static[dac_num2][channel2] + 0.015,
                       stage1_addr, underwater_addr,
                       udp_ip=udp_ip, udp_port=udp_port)



### DAC 电压参考数组 A

生成一组静态输入补偿电压。

In [264]:
comp_input_static = np.array([[1.88, 1.88, 1.88, 1.89], [1.824, 1.823, 1.817, 1.823], [1.87, 1.88, 1.88, 1.88], [1.83, 1.83, 1.83, 1.83], 
                              [1.86, 1.87, 1.87, 1.86], [1.83, 1.83, 1.83, 1.83], [1.85, 1.85, 1.85, 1.85], [1.84, 1.84, 1.84, 1.83]])
dac_value = comp_input_static - 0.04

### DAC 全通道配置测试 A

使用上一代码块的电压数组配置全部 8 个 DAC。

In [265]:
channel_map = {
    "a": 0,
    "b": 1,
    "c": 2,
    "d": 3,
}
for i in range(8):
    for j in range(4):
        config_dac128s085(i, list(channel_map.keys())[j], dac_value[i][j], 1, 1)

### DAC UART 参考值配置测试

使用旧 `uart_32ch_new.ipynb` 中的参考电压配置全部 DAC。

In [306]:
# Configure all DAC128S085 channels using the reference values from
# Veto_32ch_tdc/uart_32ch_new.ipynb.
comp_input_static_uart = np.array([
    [1.88,  1.88,  1.88,  1.89],
    [1.824, 1.823, 1.817, 1.823],
    [1.87,  1.88,  1.88,  1.88],
    [1.87,  1.84,  1.90,  1.89],
    [1.86,  1.87,  1.87,  1.86],
    [1.83,  1.83,  1.83,  1.83],
    [1.85,  1.85,  1.85,  1.85],
    [1.84,  1.84,  1.84,  1.83],
])

dac_value_uart = comp_input_static_uart + 0.05
dac_channels = ["a", "b", "c", "d"]

dac_configuration = []

for dac_num in range(8):
    for channel_index, channel in enumerate(dac_channels):
        voltage = float(dac_value_uart[dac_num][channel_index])
        config_dac128s085(
            dac_num, channel, voltage,
            1, 1
        )
        dac_configuration.append({
            "dac_num": dac_num,
            "channel": channel,
            "voltage": voltage,
        })
        print(
            f"DAC{dac_num} channel {channel.upper()}: "
            f"{voltage:.3f} V"
        )

# RBCP ACK confirms receipt; allow the final DAC SPI transaction to finish.
time.sleep(0.05)
print(f"Configured {len(dac_configuration)} DAC channels.")


DAC0 channel A: 1.930 V
DAC0 channel B: 1.930 V
DAC0 channel C: 1.930 V
DAC0 channel D: 1.940 V
DAC1 channel A: 1.874 V
DAC1 channel B: 1.873 V
DAC1 channel C: 1.867 V
DAC1 channel D: 1.873 V
DAC2 channel A: 1.920 V
DAC2 channel B: 1.930 V
DAC2 channel C: 1.930 V
DAC2 channel D: 1.930 V
DAC3 channel A: 1.920 V
DAC3 channel B: 1.890 V
DAC3 channel C: 1.950 V
DAC3 channel D: 1.940 V
DAC4 channel A: 1.910 V
DAC4 channel B: 1.920 V
DAC4 channel C: 1.920 V
DAC4 channel D: 1.910 V
DAC5 channel A: 1.880 V
DAC5 channel B: 1.880 V
DAC5 channel C: 1.880 V
DAC5 channel D: 1.880 V
DAC6 channel A: 1.900 V
DAC6 channel B: 1.900 V
DAC6 channel C: 1.900 V
DAC6 channel D: 1.900 V
DAC7 channel A: 1.890 V
DAC7 channel B: 1.890 V
DAC7 channel C: 1.890 V
DAC7 channel D: 1.880 V
Configured 32 DAC channels.


## 6. 其他调试

### 原始 TCP 连接测试

建立一个到 SiTCP 端口 24 的 TCP 连接，仅用于链路调试。

In [13]:
client = socket.socket(family=socket.AF_INET, type=socket.SOCK_STREAM)
client.settimeout(5)
client.connect(('192.168.31.16', 24))
client

<socket.socket fd=1432, family=2, type=1, proto=0, laddr=('192.168.31.12', 6157), raddr=('192.168.31.16', 24)>